# GLM
The goal of the GLM is to investigate the activation of the brain in the math and story tasks.

a) Build an appropriate experimental design matrix that you will use in the following GLM analysis.

b) Compute and plot the correlation matrix between the regressors of the design matrix and comment on the results.

c) Run the GLM at the level you deem reasonable and report the statistical maps of each of the task-related regressors. Briefly comment on the statistical maps.

d) Using the results of the GLM analysis and a contrast vector of your choice, create the activation maps corresponding to the story vs math
contrast. Please include in the report the contrast vector used and the contrast map you obtained.

e) Discuss which brain regions and networks are more activated during numerical than semantic processing, and which are more activated in the opposite contrast. As brain activity at rest can be separated into known functional networks [2], comment the results in terms of functional networks. Report known functional networks found within the statistical results and brain activation maps. You can either observe their overlaps or use brain atlas to inform your answer.

f) [Hard] Based on the literature, some regions should stand out as particularly relevant to semantic language processing. Are these regions present in your contrast maps? If not, discuss at least two possible reasons that could explain their absence.

## Loading data
The GLM needs one time series per voxel, since the preprocessed data is already 4D:

In [ ]:
import nibabel as nib
import pandas as pd

fmri_img = nib.load(path_to_preprocessed_4d)

Regarding the events, Nilearn needs a DataFrame with exactly three columns: onset (s), duration (s) and trial_type. Each unique trial_type becomes one regressor. 

In [ ]:
events = pd.read_csv(events_path)

## 3a. Building the design matrix
In order to build the GLM, first of all we need to specify EXPLAIN

We decided to add the following regressors: 
- Signal drift (slow scanner or physiological trends, modelled as low-order polynomials or cosines)
- Censoring (explain what it is)
Then it is possible to fit the model

In [ ]:
from nilearn.glm.first_level import make_first_level_design_matrix, FirstLevelModel

# Define what type of GLM we want (nature of the noise, repetition time of the data...)
fmri_glm = FirstLevelModel(t_r=7,         # repetition time: tells nilearn when each scan was acquired
			         noise_model='ar1',   # AR(1) pre-whitening for autocorrelated noise
			         standardize=False,   # keep signal units (betas in original scale)
			         hrf_model='spm',     # canonical double-gamma HRF used for convolution
			         drift_model=None,    # no drift regressors added automatically
			         high_pass=.01)       # cutoff (Hz) — only used by the 'cosine' drift model, so ignored here

from nilearn.glm.first_level.design_matrix import _make_drift
column_values, column_names = _make_drift(drift_model='polynomial',
                           frame_times=np.asarray(list(range(0, 666, 7))),  # 96 scans × TR 7 s
                                          order=3, high_pass=None)

design_mat = fmri_glm.design_matrices_[0]         # start from the task design matrix (a pandas DataFrame)
for (col, name) in zip(column_values.T, column_names):
design_mat[name] = col                        # adding a regressor = adding a column

# censoring: one regressor per outlier volume
design_mat_regged = design_mat.copy()
for i in range(10):
  reg = np.zeros((96,))*0.0
  reg[i*10] = 1.0           # in the lab, the outliers are fake (every 10th volume)
  design_mat_regged['motion outlier#' + str(i+1)] = reg

# From the events, build the design matrix X, then fits every voxel
fmri_glm = fmri_glm.fit(fmri_img, events)

# Plotting the design matrix X
from nilearn.plotting import plot_design_matrix
plot_design_matrix(fmri_glm.design_matrices_[0])

# Cancellare righe sopra??
fit_glm = FirstLevelModel(t_r=7.0, drift_model='None', minimize_memory=False) fit_glm.fit(fmri_img, design_matrices=[design_mat])

## 3b. Correlation between regressors


In [ ]:
cols = [c for c in design_mat_regged.columns if c != 'constant'] # the constant has zero variance → NaN correlations 
corr = design_mat_regged[cols].corr() 
plt.imshow(corr, vmin=-1, vmax=1, cmap='RdBu_r'); plt.colorbar() plt.xticks(range(len(cols)), cols, rotation=90); plt.yticks(range(len(cols)), cols) plt.show() 